# 00 - Filter EMG (new 40-subject dataset)

Applies the same filtering pipeline used in `EMG_OLD/Data Process-1.ipynb` to the MVC-normalized EMG columns in the new 40-subject dataset.

**Pipeline:** MVC-normalized already in source sheet -> 5 Hz Butterworth high-pass (order 4, filtfilt) -> rectify (abs) -> 10 Hz Butterworth low-pass (order 4, filtfilt) -> envelope.

**Input:** `EMG/P*_EMG.csv` (40 files, contain `EMG_MVC_CH1..8`)  
**Output:** `EMG/Filtered_40/P*_EMG_MVC_envelope.csv`

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

EMG_RAW_DIR = Path(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG")
EMG_OUT_DIR = Path(r"C:\Users\lilin\OneDrive\Desktop\ECE5424\Capstone_Project\EMG\Filtered_40")
EMG_OUT_DIR.mkdir(parents=True, exist_ok=True)

EMG_FS_DEFAULT = 500.0   # Hz
HP_CUTOFF = 5.0
LP_CUTOFF = 10.0
FILTER_ORDER = 4

MVC_CHANNELS = [f"EMG_MVC_CH{i}" for i in range(1, 9)]

In [2]:
def _safe_filtfilt(b, a, x):
    x = np.asarray(x, dtype=float)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    try:
        return filtfilt(b, a, x)
    except Exception:
        return x.copy()

def highpass(x, fs, cutoff=HP_CUTOFF, order=FILTER_ORDER):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype="highpass")
    return _safe_filtfilt(b, a, x)

def lowpass(x, fs, cutoff=LP_CUTOFF, order=FILTER_ORDER):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype="lowpass")
    return _safe_filtfilt(b, a, x)

def compute_envelope(x, fs):
    x = np.nan_to_num(np.asarray(x, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    x_hp = highpass(x, fs)
    x_rect = np.abs(x_hp)
    x_env = lowpass(x_rect, fs)
    return x_env

def estimate_fs_from_timestamps(df, timestamp_col="Timestamp"):
    # timestamps have 0.1 s precision; use rows per unique timestamp * 10
    if timestamp_col not in df.columns:
        return EMG_FS_DEFAULT
    counts = df.groupby(timestamp_col).size()
    return float(counts.median() * 10.0)

In [3]:
csv_files = sorted(EMG_RAW_DIR.glob("P*_EMG.csv"))
print(f"Found {len(csv_files)} raw EMG files.")

log_rows = []
for i, path in enumerate(csv_files, 1):
    base = path.stem  # e.g., P10_EMG
    out_path = EMG_OUT_DIR / f"{base}_MVC_envelope.csv"
    try:
        df = pd.read_csv(path)
    except Exception as e:
        print(f"[{i}/{len(csv_files)}] {path.name} -> FAIL reading: {e}")
        log_rows.append({"file": path.name, "status": "read_fail", "msg": str(e)})
        continue

    missing = [c for c in MVC_CHANNELS if c not in df.columns]
    if missing:
        print(f"[{i}/{len(csv_files)}] {path.name} -> SKIP (missing: {missing})")
        log_rows.append({"file": path.name, "status": "missing_mvc_cols", "msg": str(missing)})
        continue

    fs_est = estimate_fs_from_timestamps(df)
    fs = fs_est if 100 <= fs_est <= 5000 else EMG_FS_DEFAULT

    for ch in MVC_CHANNELS:
        df[ch] = compute_envelope(df[ch].values, fs=fs)

    df.to_csv(out_path, index=False)
    log_rows.append({
        "file": path.name,
        "status": "success",
        "n_rows": int(len(df)),
        "fs_estimated": fs_est,
        "fs_used": fs,
        "output": out_path.name,
    })
    print(f"[{i}/{len(csv_files)}] {path.name} -> {out_path.name} (rows={len(df)}, fs~{fs_est:.1f} Hz, used {fs:.1f})")

log_df = pd.DataFrame(log_rows)
log_df.to_csv(EMG_OUT_DIR / "filtering_log.csv", index=False)
print("\nDone. Log saved to:", EMG_OUT_DIR / "filtering_log.csv")

Found 40 raw EMG files.
[1/40] P10_EMG.csv -> P10_EMG_MVC_envelope.csv (rows=691500, fs~10.0 Hz, used 500.0)
[2/40] P11_EMG.csv -> P11_EMG_MVC_envelope.csv (rows=867000, fs~10.0 Hz, used 500.0)
[3/40] P12_EMG.csv -> P12_EMG_MVC_envelope.csv (rows=530000, fs~10.0 Hz, used 500.0)
[4/40] P13_EMG.csv -> P13_EMG_MVC_envelope.csv (rows=740000, fs~10.0 Hz, used 500.0)
[5/40] P14_EMG.csv -> P14_EMG_MVC_envelope.csv (rows=622500, fs~10.0 Hz, used 500.0)
[6/40] P15_EMG.csv -> P15_EMG_MVC_envelope.csv (rows=638000, fs~10.0 Hz, used 500.0)
[7/40] P16_EMG.csv -> P16_EMG_MVC_envelope.csv (rows=626500, fs~10.0 Hz, used 500.0)
[8/40] P17_EMG.csv -> P17_EMG_MVC_envelope.csv (rows=510000, fs~10.0 Hz, used 500.0)
[9/40] P18_EMG.csv -> P18_EMG_MVC_envelope.csv (rows=920000, fs~10.0 Hz, used 500.0)
[10/40] P19_EMG.csv -> P19_EMG_MVC_envelope.csv (rows=551000, fs~10.0 Hz, used 500.0)
[11/40] P1_EMG.csv -> P1_EMG_MVC_envelope.csv (rows=754500, fs~10.0 Hz, used 500.0)
[12/40] P20_EMG.csv -> P20_EMG_MVC_envelo

In [4]:
# quick sanity check on one output file
sample_path = sorted(EMG_OUT_DIR.glob("P*_MVC_envelope.csv"))[0]
sample = pd.read_csv(sample_path)
print("File:", sample_path.name)
print("Shape:", sample.shape)
print("\nMVC envelope stats per channel:")
print(sample[MVC_CHANNELS].describe().T[["mean", "std", "min", "max"]].round(3))

File: P10_EMG_MVC_envelope.csv
Shape: (691500, 29)

MVC envelope stats per channel:
              mean    std    min     max
EMG_MVC_CH1  0.531  0.615 -0.065  24.056
EMG_MVC_CH2  0.541  0.748 -0.112  23.152
EMG_MVC_CH3  0.631  0.833 -0.158  21.777
EMG_MVC_CH4  0.828  1.101 -0.168  23.840
EMG_MVC_CH5  0.701  0.981 -0.141  23.901
EMG_MVC_CH6  1.372  2.029 -0.133  38.607
EMG_MVC_CH7  0.334  0.616 -0.042  26.598
EMG_MVC_CH8  0.351  0.633 -0.074  27.680
